In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EcommerceKafkaStreaming")
    .master("local[2]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8"
    )
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.8


In [2]:
from pyspark.sql.types import (
    ArrayType,
    DoubleType,
    IntegerType,
    MapType,
    StringType,
    StructField,
    StructType
)


geo_schema = StructType([
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("ip_address", StringType(), True)
])


device_schema = StructType([
    StructField("type", StringType(), True),
    StructField("os", StringType(), True),
    StructField("browser", StringType(), True)
])


page_view_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("page_type", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category_id", StringType(), True),
    StructField("view_duration", IntegerType(), True)
])


cart_item_schema = StructType([
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True)
])


session_schema = StructType([
    StructField("session_id", StringType(), False),
    StructField("user_id", StringType(), True),
    StructField("start_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("duration_seconds", IntegerType(), True),
    StructField("geo_data", geo_schema, True),
    StructField("device_profile", device_schema, True),
    StructField(
        "viewed_products",
        ArrayType(StringType()),
        True
    ),
    StructField(
        "page_views",
        ArrayType(page_view_schema),
        True
    ),
    StructField(
        "cart_contents",
        MapType(StringType(), cart_item_schema),
        True
    ),
    StructField("conversion_status", StringType(), True),
    StructField("referrer", StringType(), True)
])

In [3]:
kafka_stream = (
    spark.readStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        "localhost:9092"
    )
    .option("subscribe", "ecommerce_sessions")

    # Use earliest for the first test.
    .option("startingOffsets", "earliest")

    # Prevent one micro-batch from becoming too large.
    .option("maxOffsetsPerTrigger", 1000)

    .load()
)

In [4]:
from pyspark.sql.functions import (
    col,
    from_json,
    size,
    to_timestamp
)


parsed_sessions = (
    kafka_stream
    .select(
        col("key").cast("string").alias("kafka_key"),
        col("value").cast("string").alias("json_value"),
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    )
    .withColumn(
        "session",
        from_json(
            col("json_value"),
            session_schema
        )
    )
    .select(
        "kafka_key",
        "topic",
        "partition",
        "offset",
        "kafka_timestamp",
        "session.*"
    )
)

In [5]:
stream_features = (
    parsed_sessions
    .withColumn(
        "session_start",
        to_timestamp("start_time")
    )
    .withColumn(
        "session_end",
        to_timestamp("end_time")
    )
    .withColumn(
        "viewed_product_count",
        size("viewed_products")
    )
    .withColumn(
        "page_view_count",
        size("page_views")
    )
    .withColumn(
        "cart_product_count",
        size("cart_contents")
    )
    .select(
        "session_id",
        "user_id",
        "session_start",
        "duration_seconds",
        col("device_profile.type").alias("device_type"),
        col("device_profile.os").alias("operating_system"),
        col("device_profile.browser").alias("browser"),
        col("geo_data.country").alias("country"),
        "viewed_product_count",
        "page_view_count",
        "cart_product_count",
        "conversion_status",
        "referrer",
        "kafka_timestamp",
        "partition",
        "offset"
    )
)

In [6]:
query = (
    stream_features.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("numRows", 10)
    .option(
        "checkpointLocation",
        "hdfs://localhost:9000/ecommerce/checkpoints/session_console"
    )
    .trigger(processingTime="5 seconds")
    .start()
)

query.awaitTermination()

StreamingQueryException: [STREAM_FAILED] Query [id = c2b52c46-68c7-4a1b-83ac-e39b2a71d848, runId = ac3f5d72-7487-4eb6-91ae-851f2e7aa8ed] terminated with exception: org.apache.kafka.common.errors.TimeoutException: Timed out waiting for a node assignment. Call: describeTopics